In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import ipywidgets as widgets
from IPython.display import display
import os

# Set renderer (helps with visibility in different Jupyter environments)
import plotly.io as pio
pio.renderers.default = "notebook"

In [ ]:
# 1. Load Data
df = pd.read_csv('data/FullJoin3_with_climate_ratings-proofed.csv')

# 2. Rename columns
rename_dict = {
    'climate_rating_current_sheet': 'Current',
    'climate_rating_emissions_limited_2050': '2050',
    'climate_rating_business_as_usual_2090': '2090',
    'climate_rating_bau_plus_1degree_2090': '2090_plus_1_degree',
    'GenusSpecies': 'Taxon',
}

# 3. Standardize Coordinates
df['LocationCoordY_fixed'] = df[['LocationCoordX', 'LocationCoordY']].min(axis=1) # Longitude
df['LocationCoordX_fixed'] = df[['LocationCoordX', 'LocationCoordY']].max(axis=1) # Latitude

# 4. Filter Locations
exclude_locs = ['Nursery', 'Nitobe Memorial Garden', 'Nitobe']
df_clean = df[~df['LocationName'].isin(exclude_locs)].copy()
df_clean = df_clean[df_clean['LocationCoordY_fixed'] < -123.245].copy()

# 5. Apply JITTER (Strength = 0.00006 for ~7m separation)
jitter_strength = 0.00006 
np.random.seed(42) 

df_clean['LocationCoordX_Jittered'] = df_clean['LocationCoordX_fixed'] + np.random.uniform(-jitter_strength, jitter_strength, size=len(df_clean))
df_clean['LocationCoordY_Jittered'] = df_clean['LocationCoordY_fixed'] + np.random.uniform(-jitter_strength, jitter_strength, size=len(df_clean))

# 6. Prepare Final DataFrame
df_ready = df_clean.rename(columns=rename_dict)
df_ready['LocationCoordX'] = df_ready['LocationCoordX_Jittered']
df_ready['LocationCoordY'] = df_ready['LocationCoordY_Jittered']

cols_to_keep = [
    'ItemAccNoFull', 'ItemLocationCode', 
    'LocationCoordX', 'LocationCoordY', 
    'LifeForm', 'Taxon', 
    'Current', '2050', '2090', '2090_plus_1_degree'
]
final_merged_df = df_ready[cols_to_keep].copy()

# 7. Apply LifeForm Mapping
lifeform_mapping = {
    'Shrub': 'Woody', 'Tree': 'Woody', 'Shrub or Tree': 'Woody',
    'Climber_Liana_Vine': 'Woody', 'Herbaceous Perennial': 'Perennial',
    'Bulb, Corm, or Tuber': 'Perennial', 'Annual': 'Short-lived',
    'Biennial': 'Short-lived', 'Habit unknown': 'Unknown'
}
final_merged_df['LifeForm'] = final_merged_df['LifeForm'].map(lifeform_mapping).fillna('Unknown')

# 8. Melt to Long Format
long_format_df = pd.melt(
    final_merged_df,
    id_vars=['ItemAccNoFull', 'ItemLocationCode', 'LocationCoordX', 'LocationCoordY', 'Taxon', 'LifeForm'],
    value_vars=['Current', '2050', '2090', '2090_plus_1_degree'],
    var_name='Era',
    value_name='ClimateRating'
)
long_format_df = long_format_df.dropna(subset=['ClimateRating'])

print(f"Data Ready: {len(long_format_df)} points (Jittered & Cleaned).")

In [ ]:
# 1. Calculate the Garden's "Center of Gravity"
# We do this once so the camera stays locked in place while you toggle options
center_lat = long_format_df['LocationCoordX'].mean()
center_lon = long_format_df['LocationCoordY'].mean()

# 2. Define the Update Function
def update_map(era, lifeform):
    subset = long_format_df[
        (long_format_df['Era'] == era) & 
        (long_format_df['LifeForm'] == lifeform)
    ].copy()
    
    fig = px.scatter_map(
        subset,
        lat="LocationCoordX",
        lon="LocationCoordY",
        color="ClimateRating",
        color_continuous_scale=[(0.0, "red"), (0.5, "yellow"), (1.0, "green")],
        range_color=[0, 11],
        hover_name="Taxon",
        hover_data={
            "LocationCoordX": False, "LocationCoordY": False,
            "ItemAccNoFull": True, "ClimateRating": True, "Era": False
        },
        map_style="open-street-map",
        
        # --- NEW: LIMITING THE SCOPE ---
        # Center the camera on the garden
        center={"lat": center_lat, "lon": center_lon},
        
        # Zoom level: 1 (World) to 20 (House). 
        # 16.5 is a tight view of the garden paths.
        zoom=16.5, 
        
        height=700,
        title=f"UBCBG Risk Map: {lifeform} Plants in {era}"
    )
    
    fig.update_traces(marker=dict(size=6, opacity=0.8))
    fig.update_layout(margin={"r":0,"t":40,"l":0,"b":0})
    fig.show()

# 3. Create Controls
era_widget = widgets.Dropdown(
    options=['Current', '2050', '2090', '2090_plus_1_degree'],
    value='2090_plus_1_degree',
    description='Era:',
)

lifeform_widget = widgets.Dropdown(
    options=list(long_format_df['LifeForm'].unique()),
    value='Woody',
    description='LifeForm:',
)

# 4. Launch
widgets.interact(update_map, era=era_widget, lifeform=lifeform_widget);

In [ ]:
def generate_plotly_export(era, lifeform, file_format='png'):
    subset = long_format_df[
        (long_format_df['Era'] == era) & 
        (long_format_df['LifeForm'] == lifeform)
    ].copy()
    
    # 1. Build the Figure
    fig = px.scatter_map(
        subset, 
        lat="LocationCoordX", 
        lon="LocationCoordY", 
        color="ClimateRating",
        color_continuous_scale=[(0.0, "red"), (0.5, "yellow"), (1.0, "green")],
        range_color=[0, 11],
        map_style="open-street-map",
        zoom=15, 
        height=800, # Base height
        title=f"Climate Risk: {lifeform} ({era})"
    )
    
    # 2. Tuning for Clarity
    # Increase marker opacity slightly for cleaner edges
    fig.update_traces(marker=dict(size=6, opacity=0.9))
    
    # Increase font sizes so they remain readable at high resolution
    fig.update_layout(
        title_font_size=24,
        font=dict(size=14, color="black")
    )
    
    safe_era = era.replace(' ', '_').replace('+', 'plus')
    filename = f"UBCBG_Map_{lifeform}_{safe_era}.{file_format}"
    
    # 3. The "Crispness" Settings
    if file_format == 'html':
        fig.write_html(filename)
    else:
        # Scale=6 creates an ultra-high-res image (approx 4000x3000 pixels)
        # We ensure width/height are explicitly set to prevent layout shifts
        fig.write_image(filename, scale=6, width=1000, height=800)
        
    print(f"Exported High-Res Image: {filename}")

# Try generating a PNG now - it should be crystal clear
# generate_plotly_export('2090_plus_1_degree', 'Woody', 'png')

In [ ]:
generate_plotly_export('2090_plus_1_degree', 'Woody', 'PDF')